# Load in the raw balloon data files, combine, and sort.

Author: Brian Green (briangre@stanford.edu)

In [1]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import netCDF4 as nc
import cartopy.crs as ccrs
import matplotlib.ticker as mticker

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tqdm.pandas()

%matplotlib inline

In [4]:
# The data where I've added in the ERA5 reanalysis T, N**2

file_ids = np.array(['01','02','03','04','05','06','07','08','09','10',
                     '11','12','13','14','15','16','17','18','19'])
flights = pd.DataFrame(columns = ['flight_id','time','latitude','longitude','altitude',
                                  'u','v','pressure','T_ERA5','N2_ERA5','superpressure'])

n_files = file_ids.size
for i in np.arange(n_files):
    
    # Load the data into a pandas dataframe
    print('File', i+1,'of',n_files,'files')
    flight = pd.read_feather(f'../era5_T_N2/concordiasi/Concordiasi_{file_ids[i]}_with_ERA5_Brian.feather')
    flight['flight_id'] = file_ids[i]
    flights = flights.append(flight, ignore_index=True)
    
# Sort and save
flights = flights.sort_values(by=['flight_id', 'time']).reset_index(drop=True)
flights.to_feather('Concordiasi_with_ERA5.feather') 
flights

File 1 of 19 files
File 2 of 19 files
File 3 of 19 files
File 4 of 19 files
File 5 of 19 files
File 6 of 19 files
File 7 of 19 files
File 8 of 19 files
File 9 of 19 files
File 10 of 19 files
File 11 of 19 files
File 12 of 19 files
File 13 of 19 files
File 14 of 19 files
File 15 of 19 files
File 16 of 19 files
File 17 of 19 files
File 18 of 19 files
File 19 of 19 files


,flight_id,time,latitude,longitude,altitude,u,v,pressure,T_ERA5,N2_ERA5,superpressure
0,01,2010-09-23 01:18:30,-78.326805,167.971268,13439.4,3.02,-11.33,124.103,193.928830,0.000451,-0.18
1,01,2010-09-23 01:19:00,-78.329597,167.976532,13572.4,3.79,-10.69,121.161,193.786223,0.000456,-0.17
2,01,2010-09-23 01:19:30,-78.331955,167.982864,13715.5,4.09,-9.96,118.220,193.641636,0.000460,-0.15
3,01,2010-09-23 01:20:00,-78.334328,167.989853,13852.7,3.93,-9.27,115.338,193.501646,0.000465,-0.15
4,01,2010-09-23 01:20:30,-78.336914,167.995972,13980.4,3.43,-8.77,112.469,193.367530,0.000470,-0.15
...,...,...,...,...,...,...,...,...,...,...,...
2526505,19,2010-11-23 09:56:30,-76.329704,-20.570866,17358.1,10.33,6.50,66.584,207.932783,0.000660,11.60
2526506,19,2010-11-23 09:57:00,-76.327957,-20.559189,17359.9,10.30,6.47,66.571,207.936609,0.000660,11.62
2526507,19,2010-11-23 09:57:30,-76.326241,-20.547585,17361.2,10.28,6.44,66.583,207.933077,0.000660,11.63
2526508,19,2010-11-23 09:58:00,-76.324539,-20.535913,17363.8,10.27,6.41,66.555,207.941321,0.000660,11.65


In [2]:
# The data from Aurelien

#file_ids = np.array(['01','02','03','04','05','06','07','08','09','10',
#                     '11','12','13','14','15','16','17','18','19'])
# Flight 15 doesn't have any ERA-5 reanalysis data
file_ids = np.array(['01','02','03','04','05','06','07','08','09','10',
                     '11','12','13','14','16','17','18','19'])
flights = pd.DataFrame(columns = ['flight_id','time','latitude','longitude','altitude',
                                  'u','v','pressure','T_ERA5','N2_ERA5','superpressure'])

n_files = file_ids.size
for i in np.arange(n_files):
    
    # Load the data into a pandas dataframe
    print('File', i+1,'of',n_files,'files')
    file_path = str(f'../concordiasi_from_aurelien/Concordiasi_{file_ids[i]}_with_ERA5.nc')
    dataset = nc.Dataset(file_path)
    flight = pd.DataFrame(columns = flights.columns)
    flight['u'] = np.array(dataset.variables['u'])
    flight['v'] = np.array(dataset.variables['v'])
    flight['altitude'] = np.array(dataset.variables['alt'])
    flight['pressure'] = np.array(dataset.variables['pressure'])
    flight['T_ERA5'] = np.array(dataset.variables['T_ERA5'])
    flight['N2_ERA5'] = np.array(dataset.variables['N2_ERA5'])
    flight['latitude'] = np.array(dataset.variables['lat'])
    flight['longitude'] = np.array(dataset.variables['lon'])
    flight['superpressure'] = np.array(dataset.variables['superpressure'])
    flight['flight_id'] = file_ids[i]
    
    # Get the time in the right format
    time = pd.DataFrame({'year': dataset.variables['year'][:],
                         'month': dataset.variables['month'][:],
                         'day': dataset.variables['day'][:],
                         'hour': dataset.variables['hour'][:],
                         'minute': dataset.variables['minute'][:],
                         'second': dataset.variables['second'][:]})
    flight['time'] = pd.to_datetime(time)
    
    # Append to the combined dataframe
    flights = flights.append(flight, ignore_index=True)
    
# Load in the Flight 15 data that I've added ERA-5 reanalysis data to
flight = pd.read_feather('../era5_T_N2/concordiasi_flight15/Concordiasi_15_with_ERA5_Brian.feather')
flights = flights.append(flight, ignore_index=True)

# Sort and save
flights = flights.sort_values(by=['flight_id', 'time']).reset_index(drop=True)
flights.to_feather('Concordiasi_with_ERA5.feather') 
flights

File 1 of 19 files


FileNotFoundError: [Errno 2] No such file or directory: b'../era5_T_N2/concordiasi/Concordiasi_01_with_ERA5_Brian.nc'